In [24]:
import pandas as pd
from sqlalchemy import create_engine
import time
import requests
import pandas as pd

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import numpy as np
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL



DB_CONFIG = {
    "host": "aws-0-eu-west-1.pooler.supabase.com",
    "port": 6543,
    "database": "postgres",
    "user": "postgres.palaoxigjbvbrgdnbstd",
    "password": "@ghZ2XjxsdL_*m*"
}



url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_CONFIG["user"],
    password=DB_CONFIG["password"],
    host=DB_CONFIG["host"],
    port=DB_CONFIG["port"],
    database=DB_CONFIG["database"]
)

engine = create_engine(url)

In [25]:
import pandas as pd


#demo istat
query_demo = """
select * from storica_demo
"""


query_anagrafica_comuni = """
select * from anagrafica_comuni
"""
query_centri = """
select * from accessibilita_centri_urbani"""

centri_urbani = pd.read_sql(query_centri, engine)

demo = pd.read_sql(query_demo, engine)

anag = pd.read_sql(query_anagrafica_comuni, engine)



In [26]:
print(anag.shape)

(7894, 42)


In [27]:
anag.columns

Index(['codice_regione',
       'Codice dell'Unità territoriale sovracomunale \n(valida a fini s',
       'codice_provincia', 'Progressivo del Comune (2)', 'codice_comune',
       'Denominazione (Italiana e straniera)', 'Denominazione in italiano',
       'Denominazione altra lingua', 'Codice Ripartizione Geografica',
       'Ripartizione geografica', 'Denominazione Regione',
       'Denominazione_province',
       'Tipologia di Unità territoriale sovracomunale ',
       'Flag Comune capoluogo di Provincia/Città metropolitana/libero ',
       'Sigla automobilistica', 'Codice Comune formato numerico',
       'Codice Comune numerico con 107 Province (dal 2017 al 2025)',
       'Codice Comune numerico con 110 Province (dal 2010 al 2016)',
       'Codice Comune numerico con 107 Province (dal 2006 al 2009)',
       'Codice Comune numerico con 103 Province (dal 1995 al 2005)',
       'Codice Catastale del Comune', 'Codice NUTS1 2021',
       'Codice NUTS2 2021 (3) ', 'Codice NUTS3 2021', 'Co

In [28]:
anag = anag[['Denominazione in italiano','0-299 (%)', '300-599 (%)', '600-899 (%)', '900-1199 (%)',
       '1200-1499 (%)', '1500-1999 (%)', '2000-2499 (%)', '>=2500 (%)',
       'comune_montano', 'Fascia_Prevalente', 'Classificazione_Finale','latitudine']]

In [29]:


demo["comune_norm"] = (
    demo["comune_norm"]
    .str.upper()
    .str.strip()
)



anag["comune_norm"] = ( anag["Denominazione in italiano"]
    .str.upper()
    .str.strip())

centri_urbani["comune_norm"] = ( centri_urbani["comune"]
    .str.upper()
    .str.strip())

In [30]:
centri_urbani.head()

,regione,provincia,comune,popolazione,hub_15min_5000ab,hub_5000_vicino,hub_5000_pop,tempo_hub_5000,comune_norm
0,Lombardia,VA,Agra,381.0,1,Luino,14125.0,14,AGRA
1,Lombardia,VA,Albizzate,5184.0,8,Albizzate,5184.0,0,ALBIZZATE
2,Lombardia,VA,Angera,5329.0,2,Angera,5329.0,0,ANGERA
3,Lombardia,VA,Arcisate,9916.0,2,Arcisate,9916.0,0,ARCISATE
4,Lombardia,VA,Arsago Seprio,4735.0,11,Casorate Sempione,5596.0,5,ARSAGO SEPRIO


In [31]:
###per comuni con slash e doppia lingua
def clean_comune(x):
    return (
        x.upper()
         .split("/")[0]
         .strip()
    )

In [32]:
centri_urbani = centri_urbani.drop_duplicates(subset=["comune"])

In [33]:
centri_urbani = centri_urbani.dropna()

In [34]:
centri_urbani.drop('comune', axis=1, inplace=True)

In [35]:
df = (

        demo

    .merge(
        anag,
        on=["comune_norm"],
        how="inner"
    )
  .merge(centri_urbani, on=["comune_norm"], how="inner")
)

In [36]:
df.columns

Index(['id_demo', 'comune', 'provincia_x', 'anno', 'm_anno_0', 'f_anno_0',
       't_anno_0', 'm_0_5_prima_infanzia', 'm_6_11_infanzia',
       'm_12_17_adolescenza', 'm_18_24_giovani_adulti',
       'm_25_34_adulti_giovani', 'm_35_44_adulti', 'm_45_54_adulti_maturi',
       'm_55_64_pre_senior', 'm_65_74_senior', 'm_75_e_oltre_anziani',
       'f_0_5_prima_infanzia', 'f_6_11_infanzia', 'f_12_17_adolescenza',
       'f_18_24_giovani_adulti', 'f_25_34_adulti_giovani', 'f_35_44_adulti',
       'f_45_54_adulti_maturi', 'f_55_64_pre_senior', 'f_65_74_senior',
       'f_75_e_oltre_anziani', 't_0_5_prima_infanzia', 't_6_11_infanzia',
       't_12_17_adolescenza', 't_18_24_giovani_adulti',
       't_25_34_adulti_giovani', 't_35_44_adulti', 't_45_54_adulti_maturi',
       't_55_64_pre_senior', 't_65_74_senior', 't_75_e_oltre_anziani',
       'totale_generale', 'delta_pct_acumulado', 'codice_comune',
       'codice_comune_normalizzato', 'comune_norm',
       'Denominazione in italiano', '0-299 

In [37]:
print(df.shape)
print(df["comune_norm"].nunique())
print(df["anno"].min(), df["anno"].max())

(173297, 62)
4982
1992 2026


# imposto comune montano == 1

In [38]:
df = df[df['comune_montano']==1]

In [39]:
df.duplicated(["comune_norm", "anno","provincia_x"]).sum() #check dei duplicati

np.int64(280)

In [40]:
dup = df[df.duplicated(["comune_norm", "anno","provincia_x"], keep=False)]

dup.sort_values(["comune_norm", "anno","provincia_x"])

,id_demo,comune,provincia_x,anno,m_anno_0,f_anno_0,t_anno_0,m_0_5_prima_infanzia,m_6_11_infanzia,m_12_17_adolescenza,...,Fascia_Prevalente,Classificazione_Finale,latitudine,regione,provincia_y,popolazione,hub_15min_5000ab,hub_5000_vicino,hub_5000_pop,tempo_hub_5000
24294,016065_1992,Castro,Bergamo,1992,7,9,16,26,28,44,...,0-299 (%),Montano,5.072399e+06,Lombardia,BG,1202.0,3,Lovere,5004.0,2
24295,016065_1992,Castro,Bergamo,1992,7,9,16,26,28,44,...,0-299 (%),Montano,4.471412e+06,Lombardia,BG,1202.0,3,Lovere,5004.0,2
24296,075096_1992,Castro,Lecce,1992,10,16,26,73,82,99,...,0-299 (%),Montano,5.072399e+06,Lombardia,BG,1202.0,3,Lovere,5004.0,2
24297,075096_1992,Castro,Lecce,1992,10,16,26,73,82,99,...,0-299 (%),Montano,4.471412e+06,Lombardia,BG,1202.0,3,Lovere,5004.0,2
24290,016065_1993,Castro,Bergamo,1993,7,4,11,33,28,38,...,0-299 (%),Montano,5.072399e+06,Lombardia,BG,1202.0,3,Lovere,5004.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141882,022165_2025,Samone,Trento,2025,3,0,3,13,14,12,...,600-899 (%),Alpi Orientali,5.107107e+06,Piemonte,TO,1519.0,3,Ivrea,22532.0,9
141875,001235_2026,Samone,Torino,2026,5,5,10,30,35,50,...,0-299 (%),Montano,5.033319e+06,Piemonte,TO,1519.0,3,Ivrea,22532.0,9
141876,001235_2026,Samone,Torino,2026,5,5,10,30,35,50,...,600-899 (%),Alpi Orientali,5.107107e+06,Piemonte,TO,1519.0,3,Ivrea,22532.0,9
141877,022165_2026,Samone,Trento,2026,1,1,2,11,15,15,...,0-299 (%),Montano,5.033319e+06,Piemonte,TO,1519.0,3,Ivrea,22532.0,9


In [41]:
df.drop('latitudine', axis=1, inplace=True)

In [42]:
dup = df[df.duplicated(["comune_norm", "anno","provincia_x"], keep=False)]
dup.sort_values(["comune_norm", "anno","provincia_x"])

,id_demo,comune,provincia_x,anno,m_anno_0,f_anno_0,t_anno_0,m_0_5_prima_infanzia,m_6_11_infanzia,m_12_17_adolescenza,...,comune_montano,Fascia_Prevalente,Classificazione_Finale,regione,provincia_y,popolazione,hub_15min_5000ab,hub_5000_vicino,hub_5000_pop,tempo_hub_5000
24294,016065_1992,Castro,Bergamo,1992,7,9,16,26,28,44,...,1,0-299 (%),Montano,Lombardia,BG,1202.0,3,Lovere,5004.0,2
24295,016065_1992,Castro,Bergamo,1992,7,9,16,26,28,44,...,1,0-299 (%),Montano,Lombardia,BG,1202.0,3,Lovere,5004.0,2
24296,075096_1992,Castro,Lecce,1992,10,16,26,73,82,99,...,1,0-299 (%),Montano,Lombardia,BG,1202.0,3,Lovere,5004.0,2
24297,075096_1992,Castro,Lecce,1992,10,16,26,73,82,99,...,1,0-299 (%),Montano,Lombardia,BG,1202.0,3,Lovere,5004.0,2
24290,016065_1993,Castro,Bergamo,1993,7,4,11,33,28,38,...,1,0-299 (%),Montano,Lombardia,BG,1202.0,3,Lovere,5004.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141882,022165_2025,Samone,Trento,2025,3,0,3,13,14,12,...,1,600-899 (%),Alpi Orientali,Piemonte,TO,1519.0,3,Ivrea,22532.0,9
141875,001235_2026,Samone,Torino,2026,5,5,10,30,35,50,...,1,0-299 (%),Montano,Piemonte,TO,1519.0,3,Ivrea,22532.0,9
141876,001235_2026,Samone,Torino,2026,5,5,10,30,35,50,...,1,600-899 (%),Alpi Orientali,Piemonte,TO,1519.0,3,Ivrea,22532.0,9
141877,022165_2026,Samone,Trento,2026,1,1,2,11,15,15,...,1,0-299 (%),Montano,Piemonte,TO,1519.0,3,Ivrea,22532.0,9


In [43]:
df = df.sort_values(["comune_norm", "anno"]) #ordino per comune/anno

In [44]:
#crescita annuale: calcola la variazione percentuale annuale rispetto all'anno precedente
df["growth_rate"] = (
    df.groupby("comune_norm")["totale_generale"]
    .pct_change()
)

In [45]:
MIN_POP_BASE = 50

pop1992 = (
    df[df["anno"] == 1992][["comune_norm", "totale_generale"]]
    .rename(columns={"totale_generale": "pop1992"})
)

df = df.merge(pop1992, on="comune_norm", how="left")

df["growth_since_1992"] = np.where(
    df["pop1992"] >= MIN_POP_BASE,
    (df["totale_generale"] - df["pop1992"]) / df["pop1992"],
    np.nan
)

In [46]:
pop1992

,comune_norm,pop1992
80173,ABBADIA LARIANA,2997
128904,ABBADIA SAN SALVATORE,7230
114496,ABRIOLA,2051
64112,ACCADIA,3088
51974,ACCEGLIO,237
...,...,...
42177,ZONE,1137
21985,ZOPPÈ DI CADORE,308
33114,ZUBIENA,1136
154866,ZUGLIO,680


In [47]:

#df = df[df['comune_montano']==1]

## qui metto la soglia di spopolamento (qui 5%)

In [48]:
df["spopolamento"] = (df["growth_since_1992"] < -0.05).astype(int)
##variabili demo
df["anziani"] = df["t_65_74_senior"] + df["t_75_e_oltre_anziani"]

df["giovani"] = df["t_18_24_giovani_adulti"] + df["t_25_34_adulti_giovani"]

df["bambini"] = (
        df["t_0_5_prima_infanzia"] +
        df["t_6_11_infanzia"] +
        df["t_12_17_adolescenza"]
)

df["share_anziani"] = df["anziani"] / df["totale_generale"]
df["share_giovani"] = df["giovani"] / df["totale_generale"]
#df["indice_vecchiaia"] = df["anziani"] / df["bambini"]

In [49]:
target = df[df["anno"] == 2025]

target

,id_demo,comune,provincia_x,anno,m_anno_0,f_anno_0,t_anno_0,m_0_5_prima_infanzia,m_6_11_infanzia,m_12_17_adolescenza,...,tempo_hub_5000,growth_rate,pop1992,growth_since_1992,spopolamento,anziani,giovani,bambini,share_anziani,share_giovani
33,097001_2025,Abbadia Lariana,Lecco,2025,7,13,20,65,65,75,...,6,-0.007217,2997.0,0.055722,0,846,532,412,0.267383,0.168142
68,052001_2025,Abbadia San Salvatore,Siena,2025,16,11,27,106,113,166,...,0,-0.003753,7230.0,-0.155602,1,1900,1067,727,0.311220,0.174775
103,076001_2025,Abriola,Potenza,2025,1,1,2,17,20,22,...,22,-0.034673,2051.0,-0.402730,1,369,203,128,0.301224,0.165714
138,071001_2025,Accadia,Foggia,2025,11,7,18,56,52,68,...,28,0.005445,3088.0,-0.282383,1,613,373,325,0.276625,0.168321
173,004001_2025,Acceglio,Cuneo,2025,1,0,1,7,1,1,...,43,0.031447,237.0,-0.308017,1,53,21,22,0.323171,0.128049
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79716,017205_2025,Zone,Brescia,2025,1,3,4,10,13,32,...,20,0.006829,1137.0,-0.092348,1,313,162,103,0.303295,0.156977
79751,025069_2025,Zoppè di Cadore,Belluno,2025,1,0,1,1,0,1,...,33,-0.052632,308.0,-0.415584,1,76,25,8,0.422222,0.138889
79786,096082_2025,Zubiena,Biella,2025,3,0,3,15,20,22,...,19,-0.009683,1136.0,-0.009683,0,372,174,113,0.330667,0.154667
79821,030136_2025,Zuglio,Udine,2025,1,1,2,7,9,14,...,9,0.005607,680.0,-0.208824,1,166,86,55,0.308550,0.159851


In [50]:
analysis_df = target.copy()

In [51]:
analysis_df.columns

Index(['id_demo', 'comune', 'provincia_x', 'anno', 'm_anno_0', 'f_anno_0',
       't_anno_0', 'm_0_5_prima_infanzia', 'm_6_11_infanzia',
       'm_12_17_adolescenza', 'm_18_24_giovani_adulti',
       'm_25_34_adulti_giovani', 'm_35_44_adulti', 'm_45_54_adulti_maturi',
       'm_55_64_pre_senior', 'm_65_74_senior', 'm_75_e_oltre_anziani',
       'f_0_5_prima_infanzia', 'f_6_11_infanzia', 'f_12_17_adolescenza',
       'f_18_24_giovani_adulti', 'f_25_34_adulti_giovani', 'f_35_44_adulti',
       'f_45_54_adulti_maturi', 'f_55_64_pre_senior', 'f_65_74_senior',
       'f_75_e_oltre_anziani', 't_0_5_prima_infanzia', 't_6_11_infanzia',
       't_12_17_adolescenza', 't_18_24_giovani_adulti',
       't_25_34_adulti_giovani', 't_35_44_adulti', 't_45_54_adulti_maturi',
       't_55_64_pre_senior', 't_65_74_senior', 't_75_e_oltre_anziani',
       'totale_generale', 'delta_pct_acumulado', 'codice_comune',
       'codice_comune_normalizzato', 'comune_norm',
       'Denominazione in italiano', '0-299 

In [52]:
##provo a fare analisi con questo
vars_to_check = [

    "share_anziani",
'0-299 (%)', '300-599 (%)', '600-899 (%)',
       '900-1199 (%)', '1200-1499 (%)', '1500-1999 (%)', '2000-2499 (%)',
       #'>=2500 (%)',
    'hub_15min_5000ab',
       'hub_5000_pop', 'tempo_hub_5000','popolazione',"growth_since_1992"
]

In [53]:
ml_df = analysis_df[vars_to_check].copy()
ml_df.columns

Index(['share_anziani', '0-299 (%)', '300-599 (%)', '600-899 (%)',
       '900-1199 (%)', '1200-1499 (%)', '1500-1999 (%)', '2000-2499 (%)',
       'hub_15min_5000ab', 'hub_5000_pop', 'tempo_hub_5000', 'popolazione',
       'growth_since_1992'],
      dtype='object')

In [54]:
ml_df = ml_df.dropna()

In [55]:
ml_df = ml_df.drop_duplicates()

In [56]:
ml_df= ml_df[[
    "share_anziani",
    "0-299 (%)",

    "hub_15min_5000ab",
    "hub_5000_pop",
    "tempo_hub_5000",
    "popolazione",'growth_since_1992'
]]

In [57]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso

In [58]:
df_train, df_test = train_test_split(ml_df, test_size=0.3, random_state=42)

In [59]:
X_train = df_train[["share_anziani",
    "0-299 (%)",

    "hub_15min_5000ab",
    "hub_5000_pop",
    "tempo_hub_5000",
    "popolazione"]].values
y_train = df_train["growth_since_1992"].values

X_test = df_test[["share_anziani",
    "0-299 (%)",

    "hub_15min_5000ab",
    "hub_5000_pop",
    "tempo_hub_5000",
    "popolazione"]].values
y_test = df_test["growth_since_1992"].values


scaler = StandardScaler()

X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

#reg_std = LinearRegression()
#reg_std.fit(X_train_std, y_train)


## Linear Regression
reg = LinearRegression()
reg.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [60]:
y_pred = reg.predict(X_test)
print('R2: %.3f' % r2_score(y_test, y_pred))
print('MSE: %.3f' % mean_squared_error(y_test, y_pred))
print('MAE: %.3f' % mean_absolute_error(y_test, y_pred))


R2: 0.343
MSE: 0.070
MAE: 0.138


In [61]:
#faccio prova con dummy
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_error

dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

print("Dummy R2:", r2_score(y_test, y_pred_dummy))
print("Dummy MAE:", mean_absolute_error(y_test, y_pred_dummy))

Dummy R2: -0.0004499181400199337
Dummy MAE: 0.20643141813356156


In [62]:
# vedo che il dummy fa r2 quasi 0: significa che il modello è significativamente più spiegante

In [63]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

reg_std = LinearRegression()
reg_std.fit(X_train_std, y_train)

pd.DataFrame({
    "variabile": [
        "share_anziani",
        "0-299 (%)",
        "hub_15min_5000ab",
        "hub_5000_pop",
        "tempo_hub_5000",
        "popolazione"
    ],
    "beta_std": reg_std.coef_
}).sort_values("beta_std", key=abs, ascending=False)

,variabile,beta_std
0,share_anziani,-0.171012
2,hub_15min_5000ab,0.028102
4,tempo_hub_5000,-0.025893
5,popolazione,-0.012135
3,hub_5000_pop,0.009686
1,0-299 (%),0.008472


## provo con RF

In [64]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    min_samples_leaf=5
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("R2:", r2_score(y_test, y_pred_rf))
print("MAE:", mean_absolute_error(y_test, y_pred_rf))

R2: 0.3868205773407466
MAE: 0.13319651956964335


In [65]:
rf.feature_importances_

array([0.74585613, 0.03403245, 0.00969071, 0.05346471, 0.066164  ,
       0.09079201])

##
### share_anziani       81.1%
### popolazione          6.2%
### hub_5000_pop         5.2%
### tempo_hub_5000       3.1%
### 0-299 (%)            2.4%
### hub_15min_5000ab     1.9%